# CLIPSeg 품질검수 — SAM3 대조 실험

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

`sam3-condition-check/sam3_qc_test.ipynb` 와 **같은 이미지 · 같은 프롬프트 · 같은 후처리**로
CLIPSeg(`CIDAS/clipseg-rd64-refined`)를 돌려 두 모델을 나란히 비교한다.
CLIPSeg 는 현행 프로덕션 백엔드(`worker/models/clipseg/`)이므로, SAM3 채택 여부는
"현행 대비 나은가"로 판단해야 한다.

### 비교가 성립하려면

프롬프트·필터·판정 로직을 노트북마다 복사해 두면 한쪽만 고쳐져 조용히 어긋난다.
그래서 공유분은 전부 **`validation/qc_shared.py`** 한 곳에 있다.

- 이미지: `qc_shared.SHARED_IMAGE_DIR` (= `sam3-condition-check/sample_images/`)
- 프롬프트: `qc_shared.DEFECT_PROMPTS` / `BENIGN_PROMPTS` / `GARMENT_PROMPT`
- 필터·판정: `filter_by_garment()` / `suppress_benign()` / `decide_verdict()`

### 공유하지 않는 것 — 확률 임계치

003 계획의 교훈("모델마다 점수 스케일이 달라 고정 임계치를 재사용하면 안 된다")이 그대로
적용된다. SAM3 의 `pred_logits.sigmoid() * presence_logits.sigmoid()` 와 CLIPSeg 의 픽셀
sigmoid 확률은 **비교 가능한 값이 아니다.** `qc_shared.BACKEND_THRESHOLDS` 에서 백엔드별로
따로 잡는다 (SAM3 score 0.30 / CLIPSeg prob 0.45).

### 구조적 차이 — 이걸 모르면 비교를 잘못 읽는다

<empty-block/>

| | SAM3 PCS | CLIPSeg |
| --- | --- | --- |
| 출력 | 인스턴스별 mask·bbox·score | 프롬프트당 확률 히트맵 1장 |
| 인스턴스 분리 | **모델이 해 준다** | 직접 이진화 + 연결요소 추출 필요 |
| 해상도 | 1008px | 224px (미세 오염이 뭉개진다) |

CLIPSeg 는 인스턴스 개념이 없어서, 히트맵을 임계치로 자르고 연결요소를 뽑아 인스턴스로
만든다. 이 노트북은 `cv2.connectedComponentsWithStats` 를 쓴다 — 프로덕션
(`worker/models/clipseg/findings.py`)은 scipy 없이 돌리려고 순수 numpy flood-fill 을 쓰지만,
**결과는 같고** 여기서는 비교 목적상 간단한 쪽을 택했다.

### 실행 순서

SAM3 노트북을 **먼저** 끝까지 돌려 `sam3-condition-check/outputs/*_qc.json` 을 만들어 두면,
마지막 섹션에서 두 모델 결과를 한 표로 비교한다. 없으면 그 섹션만 건너뛴다.

## 0. 의존성 · 환경

SAM3 노트북과 **같은 conda 환경(`qual`)** 을 쓴다. CLIPSeg 는 게이트 저장소가 아니라
HF 로그인이 필요 없고, transformers 5.14.1 에서 정상 동작하는 것을 확인했다.

In [ ]:
%pip install -q -r ../sam3-condition-check/requirements-sam3.txt

In [ ]:
import torch

print(f"torch          : {torch.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu            : {_props.name}")
    print(f"total VRAM     : {_props.total_memory / 1024**3:.2f} GB")

## 0-1. 설정 — 프롬프트는 `qc_shared` 에서 온다

여기서 프롬프트를 새로 정의하지 않는다. 바꾸려면 `validation/qc_shared.py` 의
`ACTIVE_PROMPT_SET` 을 고친다 — 그러면 SAM3 노트북과 함께 바뀐다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")
from qc_shared import (  # noqa: E402
    ACTIVE_PROMPT_SET,
    BACKEND_THRESHOLDS,
    BENIGN_CONTAINMENT,
    BENIGN_PROMPTS,
    CROP_BEFORE_DETECT,
    DEFECT_PROMPTS,
    GARMENT_CROP_PAD_RATIO,
    GARMENT_OVERLAP_MIN,
    GARMENT_PROMPT,
    MIN_AREA_RATIO,
    RESTRICT_TO_GARMENT,
    SAM3_OUTPUT_DIR,
    SHARED_IMAGE_DIR,
    SUPPRESS_BENIGN,
    VERDICT_THRESHOLDS,
    decide_verdict,
    draw_numbered_boxes,
    filter_by_garment,
    list_images,
    pad_bbox_clamped,
    pick_largest,
    remap_instances_to_full,
    remap_mask_to_full,
    renumber,
    suppress_benign,
    to_serializable,
    use_korean_font,
)

CLIPSEG_MODEL_ID = "CIDAS/clipseg-rd64-refined"
CLIPSEG_INPUT_SIZE = 224  # worker/models/clipseg/segmenter.py 와 같은 값

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("./outputs")
PROB_THRESHOLD = BACKEND_THRESHOLDS["clipseg"]["prob"]
# 의류 존재 여부만 확인하는 관대한 임계치 — 하자 탐지(0.45)와 분리한다. 처음엔 같은 임계치를
# 썼는데, "clothing" 확률이 그 기준을 못 넘으면 의류를 아예 못 찾아 크롭·필터가 통째로
# no-op 되는 문제가 실측(SAM3 쪽)에서 나왔다. CLIPSeg 도 같은 구조라 동일하게 분리한다.
GARMENT_PROB_THRESHOLD = BACKEND_THRESHOLDS["clipseg"]["garment_prob"]

print(f"프롬프트 세트 : {ACTIVE_PROMPT_SET}  (SAM3 노트북과 공유)")
for _t, _p in DEFECT_PROMPTS.items():
    print(f"  {_t:<8}: {_p}")
print()
print(f"확률 임계     : {PROB_THRESHOLD}  (CLIPSeg 전용 — SAM3 의 0.30 과 다른 스케일)")
print(
    f"의류 게이트   : {RESTRICT_TO_GARMENT}  (존재확인 임계={GARMENT_PROB_THRESHOLD}, "
    f"재확인 최소겹침={GARMENT_OVERLAP_MIN})"
)
print(
    f"  CROP_BEFORE_DETECT: {CROP_BEFORE_DETECT}"
    f"  (True 면 크롭 여유={GARMENT_CROP_PAD_RATIO} 로 실제 크롭 후 탐지 — "
    f"SAM3 쪽 실측 회귀로 기본은 False)"
)
print(f"양성 억제     : {SUPPRESS_BENIGN}")
print(f"이미지 폴더   : {SHARED_IMAGE_DIR}")

## 0-2. 모델 로드

fp32 로 올린다. CLIPSeg 는 ~150MB 수준이라 fp16 이득이 거의 없고, SAM3 와 달리 저정밀에서
확률이 뭉개질 여지만 생긴다.

In [ ]:
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

try:
    processor = CLIPSegProcessor.from_pretrained(CLIPSEG_MODEL_ID)
    model = CLIPSegForImageSegmentation.from_pretrained(CLIPSEG_MODEL_ID).to(DEVICE).eval()
    print(f"로드 완료: {CLIPSEG_MODEL_ID}  device={DEVICE}")
except torch.cuda.OutOfMemoryError as exc:
    print(f"[CUDA OOM] {exc}")
    print("  SAM3 노트북 커널이 아직 GPU 를 점유하고 있을 수 있습니다 — 그쪽 커널을 종료하세요.")
    raise
except Exception as exc:
    print(f"[모델 로드 실패] {type(exc).__name__}: {exc}")
    raise

if DEVICE == "cuda":
    _alloc = torch.cuda.memory_allocated()
    print(f"로드 후 allocated : {_alloc / 1024**2:.1f} MB ({_alloc / 1024**3:.2f} GB)")
    print("  참고: SAM3 는 fp16 로 ~1.7GB 였다.")

## 0-3. 이미지 로드 — SAM3 와 같은 폴더

`qc_shared.list_images()` 가 `sam3-condition-check/sample_images/` 를 하위 폴더까지 훑는다.
두 노트북이 정확히 같은 목록을 같은 순서로 본다.

In [ ]:
from PIL import Image

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATHS = list_images()

if not IMAGE_PATHS:
    raise FileNotFoundError(
        f"{SHARED_IMAGE_DIR} 에 이미지가 없습니다. "
        "SAM3 노트북을 먼저 실행하거나 이미지를 직접 넣어 주세요."
    )


def load_image(path):
    return Image.open(path).convert("RGB")


print(f"이미지 {len(IMAGE_PATHS)}장:")
for _i, _p in enumerate(IMAGE_PATHS):
    print(f"  [{_i}] {_p.relative_to(SHARED_IMAGE_DIR)}")

## 1. CLIPSeg 추론 — 프롬프트 -> 확률 히트맵

`processor(...)` 를 통합 호출하지 않고 **`tokenizer` 와 `image_processor` 를 분리 호출**한다.
통합 호출은 텍스트/이미지 kwarg 가 섞여 에러가 난다 — `verify_clipseg.py`(LAB-12)와
프로덕션 `segmenter.py` 가 쓰는 것과 같은 우회다.

출력은 `(프롬프트수, 224, 224)` 확률맵이라 원본 크기로 리사이즈해서 쓴다.

In [ ]:
import numpy as np


def clipseg_probs(pil_image, prompts):
    """프롬프트 리스트 -> (n, H, W) 확률맵 (원본 해상도)."""
    text_inputs = processor.tokenizer(list(prompts), return_tensors="pt", padding=True)
    image_inputs = processor.image_processor(
        [pil_image] * len(prompts),
        size={"height": CLIPSEG_INPUT_SIZE, "width": CLIPSEG_INPUT_SIZE},
        return_tensors="pt",
    )
    inputs = {**text_inputs, "pixel_values": image_inputs["pixel_values"]}
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    if logits.ndim == 2:  # 프롬프트가 1개면 배치 차원이 사라진다
        logits = logits.unsqueeze(0)

    probs = torch.sigmoid(logits).unsqueeze(1)  # (n, 1, h, w)
    probs = torch.nn.functional.interpolate(
        probs,
        size=(pil_image.height, pil_image.width),
        mode="bilinear",
        align_corners=False,
    )
    return probs[:, 0].detach().cpu().numpy()


_probs = clipseg_probs(load_image(IMAGE_PATHS[0]), list(DEFECT_PROMPTS.values()))
print(f"probs shape : {_probs.shape}  (프롬프트수, H, W)")
for _t, _p in zip(DEFECT_PROMPTS, _probs):
    print(
        f"  {_t:<8}: max={_p.max():.4f}  mean={_p.mean():.4f}  "
        f">{PROB_THRESHOLD} 픽셀 비율={float((_p > PROB_THRESHOLD).mean()):.4%}"
    )

## 2. 히트맵 -> 인스턴스 (연결요소 추출)

CLIPSeg 에는 인스턴스 개념이 없다. 확률맵을 임계치로 이진화하고 연결요소를 뽑아 SAM3 와
같은 형태(`{no, type, prompt, bbox, confidence, mask}`)로 만든다. 이렇게 맞춰야
`qc_shared` 의 필터·판정을 그대로 태울 수 있다.

- `confidence` 는 blob 내부 확률의 **최댓값**(peak)으로 잡는다 — 프로덕션 `findings.py` 와 같다.
- `MIN_AREA_RATIO`(0.0015) 미만 blob 은 노이즈로 버린다 — `registry.py` 와 같은 값이다.

In [ ]:
import cv2


def probs_to_instances(prob_map, tag, prompt, threshold=PROB_THRESHOLD):
    """확률맵 1장 -> 인스턴스 리스트. SAM3 쪽과 같은 dict 스키마로 맞춘다."""
    binary = (prob_map > threshold).astype(np.uint8)
    total = prob_map.size
    if binary.sum() == 0:
        return []

    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=4)
    found = []
    for label in range(1, count):  # 0 은 배경
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area / total < MIN_AREA_RATIO:
            continue
        mask = (labels == label).astype(np.uint8)
        x = int(stats[label, cv2.CC_STAT_LEFT])
        y = int(stats[label, cv2.CC_STAT_TOP])
        w = int(stats[label, cv2.CC_STAT_WIDTH])
        h = int(stats[label, cv2.CC_STAT_HEIGHT])
        found.append(
            {
                "type": tag,
                "prompt": prompt,
                "bbox": [x, y, x + w, y + h],
                "confidence": round(float(prob_map[mask.astype(bool)].max()), 4),
                "mask": mask,
            }
        )
    return found


def detect_all(pil_image, prompts=DEFECT_PROMPTS, threshold=PROB_THRESHOLD):
    """하자 프롬프트 전체 -> 인스턴스 병합 리스트."""
    tags = list(prompts)
    maps = clipseg_probs(pil_image, [prompts[t] for t in tags])
    merged = []
    for tag, prob_map in zip(tags, maps):
        merged.extend(probs_to_instances(prob_map, tag, prompts[tag], threshold))
    return renumber(merged)


_demo = detect_all(load_image(IMAGE_PATHS[0]))
print(f"[{IMAGE_PATHS[0].name}] 필터 전 인스턴스 {len(_demo)}건")
for _d in _demo[:10]:
    print(f"  #{_d['no']:<2} {_d['type']:<8} conf={_d['confidence']:.4f} bbox={_d['bbox']}")

## 3. 하자 탐지 + 오탐 억제 — SAM3 와 **완전히 같은 로직**

`qc_shared.filter_by_garment()` / `suppress_benign()` / `pick_largest()` / `pad_bbox_clamped()`
/ `remap_instances_to_full()` / `CROP_BEFORE_DETECT` 를 SAM3 노트북과 그대로 공유한다. 두
노트북에서 다른 후처리를 쓰면 "모델 차이"인지 "후처리 차이"인지 구별할 수 없다.

### ⚠️ 크롭(crop-then-detect)은 실측 회귀로 기본 OFF

의류 bbox 로 이미지를 실제로 잘라 그 안에서만 탐지하는 방식을 한때 썼는데, **SAM3 쪽 실제
상품 사진 8장 검증에서 6장이 기존에 잡히던 오염·찢어짐을 못 잡거나 전부 STAIN 으로 쏠리는
회귀를 냈다**(자세한 근거는 SAM3 노트북 §3, `qc_shared.CROP_BEFORE_DETECT` 주석 참고). 그래서
`CROP_BEFORE_DETECT` 기본값은 **False** — 탐지는 항상 원본 전체 이미지에서 하고, 의류는 별도로
찾아 **결과만** 거른다.

1. `clothing` 확률맵을 **관대한 전용 임계치**(`GARMENT_PROB_THRESHOLD`, 하자 탐지용 0.45 보다
   훨씬 낮음)로 이진화해 연결요소를 뽑고, 가장 큰 것을 의류로 채택한다. 하자 탐지와 같은
   임계치를 그대로 쓰면 "clothing" 확률이 그 기준을 못 넘었을 때 의류를 아예 못 찾아 이후
   전체가 조용히 no-op 된다.
2. **탐지는 항상 원본 전체 이미지에서 한다** — 잘리지도, 맥락이 좁아지지도 않는다.
3. **결과만 의류 마스크로 거른다**(`filter_by_garment`, 60% 이상 겹쳐야 유지). 의류를 아예
   못 찾으면 **아무것도 거르지 않는다** — "하자 없음"과 "판정 실패"를 뭉개지 않기 위해서다.

`CROP_BEFORE_DETECT=True` 로 바꾸면 예전 크롭 방식으로 되돌아간다.

### 양성(benign) 억제

주름·접힘·그림자·솔기·프린트 확률맵에서도 (원본 전체 이미지에서) blob 을 뽑아, 하자가 거기에
60% 이상 잠기고 양성 쪽 confidence 가 더 높으면 떨어뜨린다.

In [ ]:
def detect_garment(pil_image, threshold=GARMENT_PROB_THRESHOLD):
    """의류 인스턴스 전체(연결요소, 관대한 전용 임계치)."""
    prob_map = clipseg_probs(pil_image, [GARMENT_PROMPT])[0]
    return probs_to_instances(prob_map, "GARMENT", GARMENT_PROMPT, threshold)


def crop_to_garment(pil_image):
    """의류 영역을 찾아 그 bbox 로 이미지를 자른다 — "의류만 세그먼트한 상태에서 탐지".

    반환: (crop_image, crop_box, garment_mask_in_crop). 못 찾으면 (원본, None, None).
    """
    picked = pick_largest(detect_garment(pil_image))
    if picked is None:
        return pil_image, None, None

    crop_box = pad_bbox_clamped(picked["bbox"], GARMENT_CROP_PAD_RATIO, pil_image.size)
    x1, y1, x2, y2 = crop_box
    crop_image = pil_image.crop((x1, y1, x2, y2))
    garment_mask_in_crop = picked["mask"][y1:y2, x1:x2]
    return crop_image, crop_box, garment_mask_in_crop


def benign_instances(pil_image, threshold=PROB_THRESHOLD):
    """주름·접힘·그림자 등 정상 영역 인스턴스."""
    tags = list(BENIGN_PROMPTS)
    maps = clipseg_probs(pil_image, [BENIGN_PROMPTS[t] for t in tags])
    found = []
    for tag, prob_map in zip(tags, maps):
        found.extend(probs_to_instances(prob_map, tag, BENIGN_PROMPTS[tag], threshold))
    return found


def garment_mask_full(pil_image):
    """크롭 없이, 원본 전체 이미지 기준 의류 마스크(가장 큰 인스턴스). 못 찾으면 None."""
    picked = pick_largest(detect_garment(pil_image))
    return picked["mask"] if picked is not None else None


def inspect_image(pil_image):
    """하자 탐지 -> 의류 게이트(사후 필터) -> 양성 억제.

    CROP_BEFORE_DETECT 로 두 방식을 고른다 — SAM3 노트북 §3 에 실측 근거가 있다. 기본(False)은
    탐지를 항상 원본 전체 이미지에서 하고 의류 마스크는 결과 필터링에만 쓴다.

    (최종, 단계별 기록) 반환. bbox·mask 는 항상 원본 이미지 좌표다.
    """
    full_size = pil_image.size  # (width, height)

    if RESTRICT_TO_GARMENT and CROP_BEFORE_DETECT:
        crop_image, crop_box, garment_mask_in_crop = crop_to_garment(pil_image)
    else:
        crop_image, crop_box, garment_mask_in_crop = pil_image, None, None
    cropped = crop_box is not None
    detect_image = crop_image if cropped else pil_image

    raw_in_crop = detect_all(detect_image)
    benigns_in_crop = benign_instances(detect_image) if SUPPRESS_BENIGN else []

    if cropped:
        raw = remap_instances_to_full(raw_in_crop, crop_box, full_size)
        benigns = remap_instances_to_full(benigns_in_crop, crop_box, full_size)
        garment_mask = remap_mask_to_full(garment_mask_in_crop, crop_box, full_size)
    else:
        raw, benigns = raw_in_crop, benigns_in_crop
        garment_mask = garment_mask_full(pil_image) if RESTRICT_TO_GARMENT else None

    after_garment, off_garment = filter_by_garment(raw, garment_mask, GARMENT_OVERLAP_MIN)
    final, on_benign = suppress_benign(after_garment, benigns, BENIGN_CONTAINMENT)

    trace = {
        "raw": len(raw),
        "off_garment": len(off_garment),
        "on_benign": len(on_benign),
        "final": len(final),
        "garment_found": garment_mask is not None,
        "cropped": cropped,
        "crop_box": crop_box,
    }
    return renumber(final), trace


print(f"이미지 {len(IMAGE_PATHS)}장 처리 시작")
print()

results_by_image = {}
traces_by_image = {}
for _path in IMAGE_PATHS:
    _final, _trace = inspect_image(load_image(_path))
    results_by_image[_path] = _final
    traces_by_image[_path] = _trace
    print(
        f"  {_path.name:<40} raw {_trace['raw']:>3} "
        f"-> 배경 제외 -{_trace['off_garment']:<3} "
        f"-> 주름·그림자 억제 -{_trace['on_benign']:<3} "
        f"=> {_trace['final']:>3}건"
    )

## 4. 이미지별 요약

In [ ]:
import pandas as pd

_rows = []
for _path, _insts in results_by_image.items():
    _t = traces_by_image[_path]
    _v, _peak = decide_verdict(_insts, VERDICT_THRESHOLDS)
    _row = {
        "image": _path.name,
        "raw": _t["raw"],
        "off_garment": -_t["off_garment"],
        "on_benign": -_t["on_benign"],
        "final": _t["final"],
        "garment_found": _t["garment_found"],
    }
    for _type in DEFECT_PROMPTS:
        _row[_type] = sum(1 for d in _insts if d["type"] == _type)
    _row["max_conf"] = round(_peak, 4)
    _row["verdict"] = _v
    _rows.append(_row)

clipseg_df = pd.DataFrame(_rows)
print(clipseg_df.to_string(index=False))
clipseg_df

## 5. 넘버링 + bbox 시각화 (한글 태그)

`qc_shared.draw_numbered_boxes()` 를 SAM3 노트북과 **동일하게** 쓴다 — confidence 내림차순
번호, 한글 유형 태그(오염/스크래치/찢어짐/구멍), 빨간 박스. 같은 함수라 그림도 눈으로 바로
비교된다. 결과는 `outputs/<파일명>_clipseg_boxes.png` 로 저장한다.

In [ ]:
import matplotlib.pyplot as plt

# 그래프 제목의 한글이 두부(□□)로 깨지지 않게 한글 폰트를 잡는다.
print(f"matplotlib 한글 폰트: {use_korean_font()}")
# 박스 그리기는 qc_shared.draw_numbered_boxes() — SAM3 노트북과 같은 함수를 쓴다.

_n = len(results_by_image)
fig, axes = plt.subplots(_n, 1, figsize=(10, 8.5 * _n), squeeze=False)
for _row, (_path, _insts) in enumerate(results_by_image.items()):
    _annotated = draw_numbered_boxes(load_image(_path), _insts)
    Image.fromarray(_annotated).save(OUTPUT_DIR / f"{_path.stem}_clipseg_boxes.png")
    axes[_row][0].imshow(_annotated)
    axes[_row][0].set_title(f"[CLIPSeg] {_path.name} — {len(_insts)} instances")
    axes[_row][0].axis("off")
plt.tight_layout()
plt.show()

## 6. 확률 히트맵 — 하자 프롬프트 vs 양성 프롬프트

CLIPSeg 의 원 출력이다. **양성 프롬프트(주름·그림자)가 하자 프롬프트와 같은 자리에서
반응하는지**를 여기서 눈으로 확인한다 — 겹친다면 억제 로직이 실제로 일할 여지가 있다는 뜻이고,
안 겹친다면 주름 오탐의 원인이 다른 데 있다는 뜻이다.

기준 이미지 1장만 본다(전체를 다 그리면 행이 너무 많아진다).

In [ ]:
TARGET_INDEX = 0
_path = IMAGE_PATHS[TARGET_INDEX]
_img = load_image(_path)

_tags = list(DEFECT_PROMPTS) + list(BENIGN_PROMPTS) + ["GARMENT"]
_texts = list(DEFECT_PROMPTS.values()) + list(BENIGN_PROMPTS.values()) + [GARMENT_PROMPT]
_maps = clipseg_probs(_img, _texts)

_cols = 4
_rows_n = (len(_tags) + _cols - 1) // _cols
fig, axes = plt.subplots(_rows_n, _cols, figsize=(4.2 * _cols, 4.4 * _rows_n), squeeze=False)

for _i, (_tag, _text, _m) in enumerate(zip(_tags, _texts, _maps)):
    ax = axes[_i // _cols][_i % _cols]
    ax.imshow(_img)
    ax.imshow(_m, cmap="jet", alpha=0.5, vmin=0, vmax=1)
    _kind = "하자" if _tag in DEFECT_PROMPTS else ("의류" if _tag == "GARMENT" else "양성")
    ax.set_title(f"[{_kind}] {_tag}\nmax={_m.max():.3f}", fontsize=9)
    ax.axis("off")

for _i in range(len(_tags), _rows_n * _cols):
    axes[_i // _cols][_i % _cols].axis("off")

plt.suptitle(f"{_path.name} — CLIPSeg 확률맵", y=1.001)
plt.tight_layout()
plt.show()

## 7. SAM3 결과와 비교

SAM3 노트북이 저장해 둔 `sam3-condition-check/outputs/<파일명>_qc.json` 을 읽어 한 표로 놓는다.
그 파일이 없으면(= SAM3 노트북을 아직 안 돌렸으면) 이 섹션은 건너뛴다.

**읽는 법** — 인스턴스 개수 자체보다 다음을 본다.

1. 두 모델이 **같은 곳**을 짚는가 (bbox 시각화를 나란히 보고 판단)
2. `TEAR` / `HOLE` 을 SAM3 가 더 잡는가 — LAB-12 가 지목한 CLIPSeg 의 약점이다
3. 필터 후에도 배경·주름 오탐이 남는 쪽은 어디인가

In [ ]:
import json

_sam3_rows = []
_missing = []
for _path in IMAGE_PATHS:
    _f = SAM3_OUTPUT_DIR / f"{_path.stem}_qc.json"
    if not _f.exists():
        _missing.append(_path.name)
        continue
    _d = json.loads(_f.read_text(encoding="utf-8"))
    _insts = _d.get("instances", [])
    _row = {"image": _path.name, "sam3_total": len(_insts), "sam3_verdict": _d.get("verdict")}
    for _type in DEFECT_PROMPTS:
        _row[f"sam3_{_type}"] = sum(1 for i in _insts if i["type"] == _type)
    _row["sam3_max_conf"] = max((i["confidence"] for i in _insts), default=0.0)
    _sam3_rows.append(_row)

if _missing:
    print(f"[알림] SAM3 결과가 없는 이미지 {len(_missing)}장: {_missing}")
    print("       sam3-condition-check/sam3_qc_test.ipynb 를 끝까지 실행하면 생성됩니다.")

if not _sam3_rows:
    print("SAM3 결과가 하나도 없어 비교를 건너뜁니다.")
    compare_df = None
else:
    _clip = clipseg_df[["image", "final", "max_conf", "verdict"]].rename(
        columns={
            "final": "clipseg_total",
            "max_conf": "clipseg_max_conf",
            "verdict": "clipseg_verdict",
        }
    )
    compare_df = _clip.merge(pd.DataFrame(_sam3_rows), on="image", how="inner")
    compare_df = compare_df[
        [
            "image",
            "clipseg_total",
            "sam3_total",
            "clipseg_max_conf",
            "sam3_max_conf",
            "clipseg_verdict",
            "sam3_verdict",
        ]
    ]
    print(compare_df.to_string(index=False))
    print()
    _agree = (compare_df["clipseg_verdict"] == compare_df["sam3_verdict"]).sum()
    print(f"판정 일치: {_agree}/{len(compare_df)}장")

compare_df

### 유형별 검출 수 비교 — `TEAR` / `HOLE` 이 핵심

In [ ]:
if compare_df is None:
    print("SAM3 결과가 없어 건너뜁니다.")
else:
    _rows = []
    for _type in DEFECT_PROMPTS:
        _clip_n = int(clipseg_df[_type].sum())
        _sam3_n = int(sum(r[f"sam3_{_type}"] for r in _sam3_rows))
        _rows.append(
            {
                "defect_type": _type,
                "prompt": DEFECT_PROMPTS[_type],
                "clipseg": _clip_n,
                "sam3": _sam3_n,
                "diff(sam3-clipseg)": _sam3_n - _clip_n,
            }
        )
    type_df = pd.DataFrame(_rows)
    print(type_df.to_string(index=False))
    print()
    print("LAB-12 결론: CLIPSeg 는 오염(STAIN)은 양호하나 찢어짐·훼손에 약하다.")
    print("-> TEAR / HOLE 행에서 sam3 가 유의미하게 크면 SAM3 채택 근거가 된다.")
    type_df

## 8. 결과 저장

In [ ]:
_summary = []
for _path, _insts in results_by_image.items():
    _v, _ = decide_verdict(_insts, VERDICT_THRESHOLDS)
    _payload = {
        "image": _path.name,
        "backend": "clipseg",
        "model_id": CLIPSEG_MODEL_ID,
        "prompt_set": ACTIVE_PROMPT_SET,
        "verdict": _v,
        "instances": to_serializable(_insts),
    }
    (OUTPUT_DIR / f"{_path.stem}_clipseg_qc.json").write_text(
        json.dumps(_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _summary.append({"image": _path.name, "verdict": _v, "instances": len(_insts)})

(OUTPUT_DIR / "clipseg_qc_summary.json").write_text(
    json.dumps(_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"저장 완료 -> {OUTPUT_DIR.resolve()}")
for _s in _summary:
    print(f"  {_s['image']:<40} {_s['verdict']:<7} {_s['instances']:>3}건")

---

## 결론 (실행 후 채울 것)

| # | 항목 | 근거 셀 | 결과 |
| --- | --- | --- | --- |
| 1 | 프롬프트 한정(`on fabric`)이 배경 오탐을 줄였는가 — `bare` 세트와 비교 | §3, §4 | |
| 2 | 의류 게이트가 걷어낸 건수 (`off_garment` 열) | §4 | |
| 3 | 양성 억제가 걷어낸 건수 (`on_benign` 열) — 주름 오탐이 실제로 줄었는가 | §4, §6 | |
| 4 | 두 모델이 같은 하자를 짚는가 | §5 vs SAM3 §5 | |
| 5 | **TEAR / HOLE 을 SAM3 가 더 잡는가** — 채택의 핵심 근거 | §7 | |
| 6 | 판정 일치율 | §7 | |
| 7 | CLIPSeg 224px 해상도로 미세 오염이 뭉개지는가 | §6 히트맵 | |

### 주의

- 임계치(CLIPSeg 0.45 / SAM3 0.30)는 **둘 다 근거 없는 초기값**이다. 개수 차이의 일부는
  모델 차이가 아니라 임계치 차이일 수 있다. 라벨링된 검증셋으로 각각 스윕해야 결론이 선다.
- 이 노트북의 blob 추출은 `cv2.connectedComponentsWithStats` 이고 프로덕션은 순수 numpy
  flood-fill 이다. 결과는 같지만 프로덕션 수치와 1:1 대조할 때는 확인이 필요하다.
- `clean fabric` 대조 정규화(`registry.clean_margin`)는 여기 **적용하지 않았다** — SAM3 쪽에
  대응물이 없어 비교가 불공정해지기 때문이다. CLIPSeg 단독 성능을 볼 때는 따로 켜서 봐야 한다.